# Deep Learning: CNN & Transfer Learning pada CIFAR-10
Tugas individu. Dua arsitektur: **CNN dari nol** dan **Transfer Learning (MobileNetV2)**.

> Aktifkan GPU di Colab: Runtime > Change runtime type > T4 GPU.

## 1. Import & Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras

SEED = 42
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]


def set_seed(seed=SEED):
    keras.utils.set_random_seed(seed)


def load_data(val_size=5000):
    """Load CIFAR-10 and split into train / validation / test."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    y_train, y_test = y_train.flatten(), y_test.flatten()
    x_val, y_val = x_train[-val_size:], y_train[-val_size:]
    x_train, y_train = x_train[:-val_size], y_train[:-val_size]
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)


def plot_samples(x, y, n=10, path=None):
    plt.figure(figsize=(14, 3))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(x[i])
        plt.title(CLASS_NAMES[y[i]], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=150)
    plt.show()


def merge_histories(*histories):
    merged = {}
    for h in histories:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged


def plot_history(history, title, path=None):
    """history: keras History or dict."""
    h = history.history if hasattr(history, "history") else history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["accuracy"], label="train")
    ax[0].plot(h["val_accuracy"], label="val")
    ax[0].set_title(f"{title} - Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(h["loss"], label="train")
    ax[1].plot(h["val_loss"], label="val")
    ax[1].set_title(f"{title} - Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=150)
    plt.show()


def evaluate_model(model, x_test, y_test, title, path=None):
    """Print test metrics, classification report, and draw confusion matrix."""
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"[{title}] Test loss: {loss:.4f} | Test accuracy: {acc:.4f}")
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {title}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=150)
    plt.show()
    return loss, acc

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
import os
os.makedirs('results', exist_ok=True)
set_seed()
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Load Dataset & Eksplorasi

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = load_data()
print('Train:', x_train.shape, '| Val:', x_val.shape, '| Test:', x_test.shape)
plot_samples(x_train, y_train, path='results/sample_images.png')

In [ ]:
# Distribusi kelas (seimbang pada CIFAR-10)
import pandas as pd
pd.Series([CLASS_NAMES[i] for i in y_train]).value_counts().plot(kind='bar', figsize=(8,3), title='Distribusi kelas (train)')
plt.show()

## 3. Definisi Model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


def build_cnn(input_shape=(32, 32, 3), num_classes=10):
    """Model 1: CNN dibangun dari nol."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)

    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="cnn_from_scratch")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def build_transfer_model(num_classes=10, image_size=96, lr=1e-3):
    """Model 2: Transfer Learning dengan MobileNetV2 (pretrained ImageNet)."""
    base = keras.applications.MobileNetV2(
        input_shape=(image_size, image_size, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False  # Phase 1: feature extraction

    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Resizing(image_size, image_size)(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)  # skala ke [-1, 1] sesuai MobileNetV2
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="mobilenetv2_transfer")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model, base


def unfreeze_for_finetuning(model, base, n_last_layers=30, lr=1e-5):
    """Phase 2: fine-tuning n layer terakhir dengan learning rate kecil."""
    base.trainable = True
    for layer in base.layers[:-n_last_layers]:
        layer.trainable = False
    # BatchNorm tetap dibekukan agar statistik pretrained tidak rusak
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

## 4. Model 1: CNN dari Nol

In [ ]:
cnn = build_cnn()
cnn.summary()

In [ ]:
def get_callbacks():
    return [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)]

h_cnn = cnn.fit(x_train, y_train, validation_data=(x_val, y_val),
                epochs=30, batch_size=64, callbacks=get_callbacks())

In [ ]:
plot_history(h_cnn, 'CNN', 'results/cnn_history.png')
cnn_loss, cnn_acc = evaluate_model(cnn, x_test, y_test, 'CNN', 'results/cnn_confusion_matrix.png')

## 5. Model 2: Transfer Learning (MobileNetV2)
**Phase 1:** base dibekukan, hanya melatih classification head.  
**Phase 2:** fine-tuning 30 layer terakhir dengan learning rate kecil.

In [ ]:
tl, base = build_transfer_model()
tl.summary()

In [ ]:
# Phase 1: feature extraction
h1 = tl.fit(x_train, y_train, validation_data=(x_val, y_val),
            epochs=8, batch_size=64, callbacks=get_callbacks())

In [ ]:
# Phase 2: fine-tuning
unfreeze_for_finetuning(tl, base, n_last_layers=30, lr=1e-5)
h2 = tl.fit(x_train, y_train, validation_data=(x_val, y_val),
            epochs=8, batch_size=64, callbacks=get_callbacks())

In [ ]:
plot_history(merge_histories(h1, h2), 'MobileNetV2', 'results/transfer_history.png')
tl_loss, tl_acc = evaluate_model(tl, x_test, y_test, 'MobileNetV2', 'results/transfer_confusion_matrix.png')

## 6. Perbandingan Model

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'Model': ['CNN (scratch)', 'MobileNetV2 (transfer learning)'],
    'Test Accuracy': [cnn_acc, tl_acc],
    'Test Loss': [cnn_loss, tl_loss],
    'Parameters': [cnn.count_params(), tl.count_params()],
})
display(df)
df.plot.bar(x='Model', y='Test Accuracy', legend=False, figsize=(6,3), ylim=(0,1), rot=0)
plt.title('Perbandingan akurasi test'); plt.show()

## 7. Contoh Prediksi

In [ ]:
idx = np.random.choice(len(x_test), 10, replace=False)
pred = np.argmax(tl.predict(x_test[idx], verbose=0), axis=1)
plt.figure(figsize=(14, 3))
for i, j in enumerate(idx):
    plt.subplot(1, 10, i+1); plt.imshow(x_test[j]); plt.axis('off')
    ok = pred[i] == y_test[j]
    plt.title(f"{CLASS_NAMES[pred[i]]}\n({CLASS_NAMES[y_test[j]]})", fontsize=8, color='green' if ok else 'red')
plt.tight_layout(); plt.show()

## 8. Kesimpulan
> Tulis dengan kata-katamu sendiri: model mana yang lebih akurat, kenapa transfer learning (tidak) lebih unggul, kelas yang sering salah diprediksi, dan indikasi overfitting dari kurva training.

In [ ]:
os.makedirs('results', exist_ok=True)
cnn.save('results/cnn_model.keras')
tl.save('results/transfer_model.keras')